In [1]:
import os

# --- CONFIGURACIÓN DE RUTAS PARA NOTEBOOK ---
# Pega aquí la ruta de tu carpeta 'data'
data_dir = r"\luisf\Documents\trabajo-California-Traffic-Collision\data"
delta_path = os.path.join(data_dir, "delta_lake")

print(f"Buscando Delta Lake en: {delta_path}")

Buscando Delta Lake en: \luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake


In [1]:
import pandas as pd
import os

# --- 1. RUTA A LA CARPETA QUE YA TIENE LA VERSIÓN 2 ---
# (Asegúrate de que la carpeta 'collisions' esté en esta ruta)
ruta_base = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\raw"

# --- 2. LEER TODOS LOS ARCHIVOS PARQUET ---
print("Leyendo archivos de la Versión 2...")

# Delta Lake guarda los datos en varios archivos .parquet, los buscamos todos:
archivos_parquet = [f for f in os.listdir(ruta_base) if f.endswith('.parquet')]

if not archivos_parquet:
    print("Error: No encontré archivos .parquet. Revisa que la ruta sea correcta.")
else:
    # Leemos y concatenamos todo en un solo DataFrame de Pandas
    lista_df = [pd.read_parquet(os.path.join(ruta_base, f)) for f in archivos_parquet]
    df_final = pd.concat(lista_df, ignore_index=True)

    print(f"¡LOGRADO! Se cargaron {len(df_final)} registros exitosamente.")
    
    # --- 3. VISTA PREVIA ---
    print("\nResumen de las primeras filas:")
    print(df_final.head())

Leyendo archivos de la Versión 2...


FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'C:\\Users\\luisf\\Documents\\trabajo-California-Traffic-Collision\\data\\raw'

In [ ]:
import pandas as pd
import os

# --- 1. ASEGÚRATE QUE ESTA RUTA SEA EXACTA ---
# Si tu carpeta en el disco C se llama diferente, cámbialo aquí.
# Ejemplo: si es "C:\Data\delta_lake", pon esa.
base_path = r"C:\Users\trabajo-California-Traffic-Collision\data\delta_lake" 

def cargar_tabla(nombre_carpeta):
    ruta = os.path.join(base_path, nombre_carpeta)
    # Verificamos si la carpeta existe antes de intentar leer
    if not os.path.exists(ruta):
        print(f" ERROR: No existe la carpeta: {ruta}")
        return pd.DataFrame()
    
    # Buscamos archivos .parquet
    archivos = [f for f in os.listdir(ruta) if f.endswith('.parquet')]
    if not archivos:
        print(f"ADVERTENCIA: No hay archivos .parquet en: {ruta}")
        return pd.DataFrame()
        
    return pd.concat([pd.read_parquet(os.path.join(ruta, f)) for f in archivos], ignore_index=True)

# Cargamos las tablas
print("Cargando datos...")
collisions = cargar_tabla("collisions")
victims_clean = cargar_tabla("involved_victims")
parties = cargar_tabla("parties")
victims_raw = cargar_tabla("victims")

# --- 2. ESTO DEBE MOSTRAR NÚMEROS MAYORES A 0 ---
if not collisions.empty:
    print(f"✅ ¡ÉXITO! Colisiones cargadas: {len(collisions)} filas")
    print("\nColumnas disponibles en Colisiones:", collisions.columns.tolist())
else:
    print("❌ Las tablas siguen vacías. Revisa la ruta 'base_path' arriba.")

Cargando datos...
❌ ERROR: No existe la carpeta: C:\Users\trabajo-California-Traffic-Collision\data\delta_lake\collisions
❌ ERROR: No existe la carpeta: C:\Users\trabajo-California-Traffic-Collision\data\delta_lake\involved_victims
❌ ERROR: No existe la carpeta: C:\Users\trabajo-California-Traffic-Collision\data\delta_lake\parties
❌ ERROR: No existe la carpeta: C:\Users\trabajo-California-Traffic-Collision\data\delta_lake\victims
❌ Las tablas siguen vacías. Revisa la ruta 'base_path' arriba.


In [5]:
# SECCIÓN: CONSULTAS DE POBLACIÓN Y RIESGO 

# Q3. Colisiones ocurridas bajo condiciones climáticas adversas (No despejado)
# Filtramos todo lo que NO sea 'clear'
q3 = collisions[collisions['weather_1'] != 'clear']
print(f"Q3: Colisiones bajo condiciones climaticas adversas: {len(q3)} filas")

# Q4. Víctimas que no portaban equipo de seguridad (cinturón, casco, etc.)
# Usamos 'victims_raw' para detalles específicos del equipo
q4 = victims_raw[victims_raw['victim_safety_equipment_1'].str.contains('none', na=False, case=False)]
print(f"Q4: Victimas sin equipo de seguridad: {len(q4)} filas")

# Q5. Accidentes ocurridos en condiciones de oscuridad (con o sin alumbrado)
q5 = collisions[collisions['lighting'].str.contains('dark', na=False, case=False)]
print(f"Q5: Colisiones en condiciones de oscuridad: {len(q5)} filas")

# Q6. Colisiones que tuvieron lugar en intersecciones
q6 = collisions[collisions['location_type'] == 'intersection']
print(f"Q6: Colisiones en intersecciones: {len(q6)} filas")

# Q7. Perfil de riesgo: Mujeres de la tercera edad (> 65 años)
# IMPORTANTE: Usamos 'victims_clean' (tu tabla sin repetidos) para estadística real
q7 = victims_clean[(victims_clean['victim_sex'] == 'female') & (victims_clean['victim_age'] > 65)]
print(f"Q7: Víctimas de la tercera edad: {len(q7)} filas")

# Q8. Accidentes donde estuvieron involucradas motocicletas
q8 = parties[parties['statewide_vehicle_type'] == 'motorcycle']
print(f"Q8: Accidentes con motocicletas: {len(q8)} filas")

# Q9. Colisiones causadas por exceso de velocidad (PCF Violation)
q9 = collisions[collisions['pcf_violation_category'] == 'speeding']
print(f"Q9: Colisiones por exceso de velocidad: {len(q9)} filas")

# Q10. Víctimas que fueron expulsadas del vehículo durante el impacto
q10 = victims_raw[victims_raw['victim_ejected'] == 'yes']
print(f"Q10: Víctimas expulsadas del vehículo: {len(q10)} filas")

KeyError: 'weather_1'

In [8]:
import os
import pandas as pd

# A1. Letalidad promedio según rangos de edad (Uso de tabla Limpia)

# Cruzamos víctimas con colisiones para obtener el conteo de fallecidos

letalidad_df = pd.merge(victims_clean, collisions[['case_id', 'killed_victims']], on='case_id')
bins_edad = [0, 25, 60, 100]

etiquetas = ['Jóvenes', 'Adultos', 'Ancianos']

print("\nA1. Promedio de fallecidos por rango de edad:")

print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())

# A2. Relación entre uso de celular y culpabilidad en el accidente
parties['at_fault'] = pd.to_numeric(parties['at_fault'], errors='coerce')

print("\nA2. Probabilidad de ser culpable según uso de celular:")

print(parties.groupby('cellphone_in_use')['at_fault'].mean())

# A3. Top 5 de marcas de vehículos con mayor frecuencia de culpabilidad

print("\nA3. Top 5 marcas de vehículos involucradas en choques por culpa:")

marcas_culpa = parties[parties['at_fault'] == 1]['vehicle_make'].value_counts().head(5)

print(marcas_culpa)

# A4. Severidad del accidente según el estado de la superficie de la vía

print("\nA4. Promedio de heridos según tipo de carretera (Road Surface):")

print(collisions.groupby('road_surface')['injured_victims'].mean())


# A5. Matriz de Sobriedad vs Severidad del Accidente

# Este análisis es fundamental para medir el impacto del alcohol en la gravedad

sobriedad_analisis = pd.merge(parties[['case_id', 'party_sobriety']], 
                              
                              collisions[['case_id', 'collision_severity']], on='case_id')
print("\nA5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque")

print(pd.crosstab(sobriedad_analisis['party_sobriety'], sobriedad_analisis['collision_severity']))

NameError: name 'victims_clean' is not defined

In [7]:
import pandas as pd
import os
import numpy as np

# Configuración para que Jupyter muestre todas las columnas
pd.set_option('display.max_columns', None)

In [6]:
# --- ANÁLISIS: CICLISTAS SOBRIOS ---
df_ciclistas = df_analisis[df_analisis['statewide_vehicle_type'].str.contains('bicycle|02', case=False, na=False)].copy()

# Creamos la tabla (Matriz de confusión)
tabla_ciclistas = pd.crosstab(
    df_ciclistas['sobriedad_limpia'], 
    df_ciclistas['victim_degree_of_injury'], 
    margins=True
)

# ESTO ES LO QUE HACE QUE SE VEA EN JUPYTER
print("TABLA: SOBRIEDAD VS LESIÓN EN CICLISTAS")
display(tabla_ciclistas) # Esto genera la tabla estilo Excel

TABLA: SOBRIEDAD VS LESIÓN EN CICLISTAS


victim_degree_of_injury,complaint of pain,killed,no injury,other visible injury,severe injury,All
sobriedad_limpia,,,,,,
Bajo Influencia,756,72,864,1440,360,3492
Desconocido/NA,576,36,72,684,72,1440
No reportado,432,0,216,720,324,1692
Sobrio,15624,144,6480,17028,2232,41508
All,17388,252,7632,19872,2988,48132


In [5]:
# --- ANÁLISIS: CONDUCTOR MOTORIZADO ---
ids_bici = df_analisis[df_analisis['statewide_vehicle_type'].str.contains('bicycle|02', case=False, na=False)]['case_id'].unique()
df_victimarios = df_analisis[(df_analisis['case_id'].isin(ids_bici)) & 
                             (~df_analisis['statewide_vehicle_type'].str.contains('bicycle|02', case=False, na=False))].copy()

tabla_cond = pd.crosstab(df_victimarios['sobriedad_limpia'], df_victimarios['victim_degree_of_injury'], margins=True)

print("TABLA: SOBRIEDAD DEL CONDUCTOR QUE ATROPELLA")
display(tabla_cond)

TABLA: SOBRIEDAD DEL CONDUCTOR QUE ATROPELLA


victim_degree_of_injury,complaint of pain,killed,no injury,other visible injury,severe injury,All
sobriedad_limpia,,,,,,
Bajo Influencia,324,36,72,468,36,936
Desconocido/NA,1116,0,216,1044,72,2448
No reportado,1548,0,36,1980,432,3996
Sobrio,13680,216,7560,14364,1692,37512
All,16668,252,7884,17856,2232,44892
